## tl;dr

原生 48/48 个进程通过固定协议的稳定性与回归检查；app 24/24 个进程完整，但五项首次耗时、一项热态 PSS 波动超限，未通过稳定性验收。全部 72 遍 app 译文与已评分输出一致。本场追加前后前台核验，不修改场景、计时范围或门槛；以下从原始记录重算，不拼接旧场次。

## Context & Methods

### Key Assumptions

- 同一台小米 14，原生八场景 × 两版本 × 三轮；app 八场景 × 三轮；每进程三遍。
- 原生使用 RSS，app 使用 PSS，分别核验，不混算。
- 前后快照不能证明持续全程前台；不倒推旧记录的锁屏状态。
- 质量复用要求新旧译文逐字及 SHA-256 一致，参考译文和 COMET checkpoint 沿用已复核记录。
- 不拼接场次、不删除失败或波动样本，不放宽 10% 门槛。

## Data

### 1. Load local, read-only verification helpers

In [1]:
from pathlib import Path
import json
import runpy
from IPython.display import Markdown, display
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'tools/app-bench/run.py').is_file())
record = root / 'benchmarks/v0.3.0/mi14-2026-09-09/charged'
checks = runpy.run_path(str(record / 'verify.py'))

## Results

### 2. Keep the aborted attempt separate

In [2]:
aborted = checks['native_checks']('native-before-foreground-check.json')
assert not aborted['accepted']
print(aborted)

{'exit_code': 2, 'accepted': False, 'successful_processes': 1, 'planned_processes': 48, 'diagnostic': 'INVALID: v0.2.0: missing or unexpected scenarios'}


### 3. Verify the guarded native session and collector snapshots

In [3]:
native = checks['native_checks']()
print(json.dumps(native, ensure_ascii=False, indent=2))
native_rows = checks['native_table']()
rows = ['| Scenario | Version | First inputs/s | Warm inputs/s | First RSS MiB |', '|---|---|---:|---:|---:|']
for r in native_rows:
    if r['complete']:
        rows.append(f"| {r['scenario']} | {r['version']} | {r['cold_inputs_per_second']:.2f} | {r['warm_inputs_per_second']:.2f} | {r['peak_rss_mib']:.1f} |")
    else:
        rows.append(f"| {r['scenario']} | {r['version']} | incomplete | incomplete | incomplete |")
display(Markdown('\n'.join(rows)))
print('Raw metric spreads exceeding 10%:')
for r in native_rows:
    for metric, value in r.get('spread_pct', {}).items():
        if value > 10:
            print(r['scenario'], r['version'], metric, round(value, 2))

{
  "exit_code": 0,
  "accepted": true,
  "successful_processes": 48,
  "planned_processes": 48,
  "diagnostic": "enzh_w1            cold_speed     62.28 -> 63.10 inputs/s (1.01x, +1.3%); raw cold_ms: 3211.165 -> 3169.449 ms\nenzh_w1            warm_speed     68.60 -> 68.66 inputs/s (1.00x, +0.1%); raw warm_ms: 2915.564 -> 2912.769 ms\nenzh_w1            peak_rss_mib     207.945 ->   207.242 (-0.3%)\nenzh_w2            cold_speed     109.10 -> 104.03 inputs/s (0.95x, -4.6%); raw cold_ms: 1833.245 -> 1922.556 ms\nenzh_w2            warm_speed     126.15 -> 125.53 inputs/s (1.00x, -0.5%); raw warm_ms: 1585.383 -> 1593.197 ms\nenzh_w2            peak_rss_mib     304.340 ->   305.699 (+0.4%)\nenzh_w4            cold_speed     158.63 -> 160.67 inputs/s (1.01x, +1.3%); raw cold_ms: 1260.795 -> 1244.756 ms\nenzh_w4            warm_speed     216.74 -> 210.21 inputs/s (0.97x, -3.0%); raw warm_ms: 922.769 -> 951.434 ms\nenzh_w4            peak_rss_mib     524.020 ->   521.051 (-0.6%)\npivot_w1  

| Scenario | Version | First inputs/s | Warm inputs/s | First RSS MiB |
|---|---|---:|---:|---:|
| enzh_w1 | v0.2.0 | 62.28 | 68.60 | 207.9 |
| enzh_w1 | v0.3.0 | 63.10 | 68.66 | 207.2 |
| enzh_w2 | v0.2.0 | 109.10 | 126.15 | 304.3 |
| enzh_w2 | v0.3.0 | 104.03 | 125.53 | 305.7 |
| enzh_w4 | v0.2.0 | 158.63 | 216.74 | 524.0 |
| enzh_w4 | v0.3.0 | 160.67 | 210.21 | 521.1 |
| pivot_w1 | v0.2.0 | 28.33 | 31.14 | 314.2 |
| pivot_w1 | v0.3.0 | 28.02 | 31.07 | 317.3 |
| enzh_b512p | v0.2.0 | 65.26 | 70.30 | 208.0 |
| enzh_b512p | v0.3.0 | 65.38 | 70.38 | 207.3 |
| pivot_b512p | v0.2.0 | 31.60 | 34.11 | 309.8 |
| pivot_b512p | v0.3.0 | 31.60 | 34.00 | 308.9 |
| enzh_w2_512p | v0.2.0 | 105.87 | 130.40 | 290.0 |
| enzh_w2_512p | v0.3.0 | 104.57 | 131.10 | 289.3 |
| pivot_w2_512p | v0.2.0 | 50.76 | 58.31 | 442.4 |
| pivot_w2_512p | v0.3.0 | 50.15 | 58.25 | 438.5 |

Raw metric spreads exceeding 10%:


### 4. Verify app integrity, acceptance and scored-output identity

In [4]:
app = checks['app_checks']()
verdict = app['verdict']
assert native['accepted'] and native['successful_processes'] == 48
assert not verdict['accepted'] and len(verdict['errors']) == 6
assert all(s['successful_processes'] == 3 for s in verdict['scenarios'])
assert app['matched_scored_passes'] == 72 and not app['needs_scoring']
print('Accepted:', verdict['accepted'])
print('Issues:', verdict['errors'])
print('Scored passes matched:', app['matched_scored_passes'])
print('Unknown output hashes needing scoring:', app['needs_scoring'])
print('COMET × 100:', app['comet_x100'])
print('Quality source SHA-256:', app['quality_source_sha256'])
print('Starting battery temperature range:', app['starting_battery_temperature_c'])
print('All excessive spreads with raw values:')
print(json.dumps(app['spreads_exceeding_10pct'], ensure_ascii=False, indent=2))
print('Single-thread / ML Kit ratios:', app['single_thread_vs_mlkit'])

Accepted: False
Issues: ['enzh/mlkit/Nonet: cold_ms spread exceeds 10%', 'enzh/bergamot/4t: cold_ms spread exceeds 10%', 'enzh/bergamot/4t: warm_pss_mib spread exceeds 10%', 'jazh/bergamot/1t: cold_ms spread exceeds 10%', 'jazh/bergamot/2t: cold_ms spread exceeds 10%', 'jazh/bergamot/4t: cold_ms spread exceeds 10%']
Scored passes matched: 72
Unknown output hashes needing scoring: []
COMET × 100: {'enzh/mlkit': [72.69342935085297], 'enzh/bergamot': [87.26716721057892], 'jazh/mlkit': [68.92763012647629], 'jazh/bergamot': [86.71308651566505]}
Quality source SHA-256: 6e3f12d4ebc1c7a0f71735307dada1ba6735f3261e91c22f3303ebb6e85e891b
Starting battery temperature range: [30.5, 32.4]
All excessive spreads with raw values:
[
  {
    "scenario": "enzh/mlkit/Nonet",
    "metric": "cold_ms",
    "values": [
      9206.706767,
      13868.735672,
      13879.288797
    ],
    "spread_pct": 33.69147801579068
  },
  {
    "scenario": "enzh/bergamot/4t",
    "metric": "cold_ms",
    "values": [
      1

### 5. Rebuild all eight app rows; consult acceptance before reuse

In [5]:
rows = ['| Scenario | First inputs/s | Warm inputs/s | First PSS MiB | Warm PSS MiB |', '|---|---:|---:|---:|---:|']
for s in verdict['scenarios']:
    if all(s.get(k) is not None for k in ('cold_inputs_per_second', 'warm_inputs_per_second', 'cold_pss_mib', 'warm_pss_mib')):
        rows.append(f"| {s['scenario']} | {s['cold_inputs_per_second']:.2f} | {s['warm_inputs_per_second']:.2f} | {s['cold_pss_mib']:.1f} | {s['warm_pss_mib']:.1f} |")
    else:
        rows.append(f"| {s['scenario']} | incomplete | incomplete | incomplete | incomplete |")
display(Markdown('\n'.join(rows)))

| Scenario | First inputs/s | Warm inputs/s | First PSS MiB | Warm PSS MiB |
|---|---:|---:|---:|---:|
| enzh/mlkit/Nonet | 14.42 | 14.58 | 185.9 | 189.6 |
| enzh/bergamot/1t | 65.52 | 86.51 | 263.0 | 232.1 |
| enzh/bergamot/2t | 114.48 | 132.64 | 376.5 | 320.3 |
| enzh/bergamot/4t | 177.12 | 246.82 | 599.2 | 600.1 |
| jazh/mlkit/Nonet | 6.82 | 6.89 | 236.9 | 238.1 |
| jazh/bergamot/1t | 44.19 | 39.77 | 353.2 | 329.7 |
| jazh/bergamot/2t | 53.82 | 61.92 | 550.4 | 505.0 |
| jazh/bergamot/4t | 88.80 | 109.86 | 931.7 | 933.6 |

## Takeaways

完整场次仍须通过独立的噪声、条件与输出检查才能作为回归基线。未通过时，表中完整行只是描述性中位数，不能以其替代通过结论。COMET 对照限于固定语料和被测模型，不证明每条译文不变，也不代表 Google Cloud Translation。